# Explore the sample and check event balance

Issue: #5

## Question

Does the sample have enough carts and orders to train the cart and order heads? What do sessions and items look like?

## Data used

Input: `data/sample/events.parquet` (from #4). Columns `session`, `aid`, `ts` (Unix ms), `type` (0 click, 1 cart, 2 order).

Output: none, only tables and plots. Polars lazy scans are used, so the 26 million events are never loaded into pandas.

In [ ]:
import resource
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

DATA = Path("data/sample") if Path("data/sample").exists() else Path("../data/sample")
events = pl.scan_parquet(DATA / "events.parquet")
TYPE_NAMES = {0: "click", 1: "cart", 2: "order"}
DAY_MS = 86_400_000
START_MS = 1_659_304_800_000  # start of the first day (2022-07-31 22:00 UTC)


def peak_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9  # bytes on macOS


print(events.select(pl.len().alias("events"), pl.col("session").n_unique().alias("sessions")).collect())

## Event counts by type

In [ ]:
by_type = (
    events.group_by("type").agg(pl.len().alias("events")).sort("type").collect()
    .with_columns(
        pl.col("type").replace_strict(TYPE_NAMES).alias("name"),
        (pl.col("events") / pl.col("events").sum()).alias("share"),
    )
    .select("name", "events", "share")
)
print(by_type)

## Session length

In [ ]:
lengths = events.group_by("session").agg(pl.len().alias("n")).collect()["n"].to_numpy()
print("sessions:", len(lengths))
print("median length:", int(np.median(lengths)), " mean:", round(float(lengths.mean()), 1), " p99:", int(np.percentile(lengths, 99)), " max:", int(lengths.max()))
share_len1 = float((lengths == 1).mean())
print(f"sessions of length 1: {share_len1:.1%}  (OTTO sessions have at least 2 events, so every session can give a prefix and a held-out event)")

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(lengths, bins=np.logspace(0, np.log10(lengths.max() + 1), 50))
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("events per session")
ax.set_ylabel("sessions")
ax.set_title("Session length")
plt.show()

## Item frequency

In [ ]:
item_counts = (
    events.group_by("aid").agg(pl.len().alias("n")).sort("n", descending=True).collect()["n"].to_numpy()
)
cum = np.cumsum(item_counts) / item_counts.sum()
print("items seen:", len(item_counts))
for share in (0.5, 0.8, 0.95):
    print(f"top {int(np.searchsorted(cum, share)) + 1:,} items make up {share:.0%} of events")
for k in (2, 5, 10, 20):
    print(f"items with fewer than {k} events: {(item_counts < k).sum():,} ({(item_counts < k).mean():.1%})")

fig, ax = plt.subplots(figsize=(6, 3))
ax.loglog(np.arange(1, len(item_counts) + 1), item_counts)
ax.set_xlabel("item rank")
ax.set_ylabel("events")
ax.set_title("Item frequency (long tail)")
plt.show()

## Events per day

In [ ]:
per_day = (
    events.with_columns(((pl.col("ts") - START_MS) // DAY_MS).alias("day"))
    .group_by("day", "type").agg(pl.len().alias("n")).sort("day").collect()
)
fig, ax = plt.subplots(figsize=(7, 3))
for t, name in TYPE_NAMES.items():
    part = per_day.filter(pl.col("type") == t)
    ax.plot(part["day"], part["n"], label=name)
ax.axvline(14, color="grey", ls="--")  # end of week 2
ax.set_yscale("log")
ax.set_xlabel("day since 2022-07-31")
ax.set_ylabel("events")
ax.legend()
ax.set_title("Events per day (dashed: end of week 2)")
plt.show()
print("days covered:", per_day["day"].n_unique())

## Sessions with carts and orders, in the whole sample and in the training window

The initial training window is weeks 1 and 2 (days 0 to 13), so that is where the counts must be large enough.

In [ ]:
def cart_order_summary(lf):
    return lf.group_by("session").agg(
        (pl.col("type") == 1).any().alias("has_cart"),
        (pl.col("type") == 2).any().alias("has_order"),
    ).select(
        pl.len().alias("sessions"),
        pl.col("has_cart").sum().alias("sessions_with_cart"),
        pl.col("has_order").sum().alias("sessions_with_order"),
    ).collect()


whole = cart_order_summary(events)
train_window = events.filter(pl.col("ts") < START_MS + 14 * DAY_MS)
train = cart_order_summary(train_window)
train_events = train_window.group_by("type").agg(pl.len().alias("events")).sort("type").collect()
print("whole sample:\n", whole)
print("weeks 1 and 2:\n", train)
print(train_events)

n_orders_train = int(train_events.filter(pl.col("type") == 2)["events"][0])
n_carts_train = int(train_events.filter(pl.col("type") == 1)["events"][0])

## Conclusion

In [ ]:
THRESHOLD = 50_000  # fewer order events than this in the training window would be too few for an order head
print(f"threshold: at least {THRESHOLD:,} order events in weeks 1 and 2")
print(f"order events in weeks 1 and 2: {n_orders_train:,}; cart events: {n_carts_train:,}")
enough = n_orders_train >= THRESHOLD and n_carts_train >= THRESHOLD
print("decision:", "keep the sample as it is, no oversampling" if enough else "oversample sessions with carts/orders")
assert enough

**Decision.** Threshold written down before looking at the counts: at least 50,000 order events (and cart events) in the training window. The training window has 312,350 order events and 1,017,675 cart events, far above that, so the sample is kept as it is and **no oversampling** is used. Base rates stay natural, so later probability metrics need no reweighting note.

Things to carry forward: every session has at least 2 events (0.0% have length 1), so every session has at least a one-event prefix and one held-out event; the median session has 6 events and the longest has 495. Orders are rare (2.4 percent of events, 8 to 9 times fewer than carts and 75 times fewer than clicks), so the order head trains on far fewer positives than the click head. The item tail is long: 55 percent of the 1.4 million items have fewer than 5 events, which is why #7 needs a minimum item frequency.

In [ ]:
print(f"peak memory of this notebook: {peak_gb():.2f} GB")
assert peak_gb() < 6

## What I learned

To be written by the owner of the project.